In [ ]:
!pip -q install -U "vllm[bench]" openai lmcache requests

In [ ]:
!wget -O /content/sharegpt.json \
  https://huggingface.co/datasets/learnanything/sharegpt_v3_unfiltered_cleaned_split/resolve/main/ShareGPT_V3_unfiltered_cleaned_split.json

In [ ]:
!nvidia-smi

In [26]:
!pkill -f vllm

In [13]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
API_KEY = "token-abc123"
HOST = "127.0.0.1"
PORT = 8000
DATASET_NAME = "sharegpt"  # "prefix_repetition"

LMCACHE_CPU_GB = "30.0"
KV_OFFLOAD_GB = "24"

BENCH_CONFIG = {
    "num_prompts": 500,
    "num_warmups": 5,
    "max_concurrency": 128,
    "temperature": 0,
}

DATASET_CONFIG = {
    "prefix_repetition": {
        "prefix_len": 16384,
        "suffix_len": 256,
        "num_prefixes": 100,
        "output_len": 512,
    },
    "sharegpt": {
        "dataset_path": "/content/sharegpt.json",
    },
}


In [14]:
import os
import time
import json
import subprocess
import requests
from pathlib import Path

BASE_URL = f"http://{HOST}:{PORT}/v1"

def kill_vllm():
    subprocess.run("pkill -f 'vllm serve'", shell=True, check=False)
    time.sleep(5)

def show_gpu():
    subprocess.run("nvidia-smi", shell=True, check=False)

def tail_log(log_file, n=80):
    subprocess.run(f"tail -n {n} {log_file}", shell=True, check=False)

def wait_for_server(timeout_s=180):
    start = time.time()
    while time.time() - start < timeout_s:
        try:
            r = requests.get(f"{BASE_URL}/models", timeout=5)
            if r.status_code == 200:
                print("vLLM server is up")
                print(r.json())
                return True
            else:
                print("waiting...", r.status_code)
        except Exception:
            pass
        time.sleep(2)
    raise RuntimeError("Server did not come up. Check the log file.")

def start_server(log_file, lmcache=False, prefix_caching=True, gpu_mem_util=0.70):
    kill_vllm()

    # Clear LMCache env vars first
    for k in [
        "LMCACHE_USE_EXPERIMENTAL",
        "LMCACHE_LOCAL_CPU",
        "LMCACHE_MAX_LOCAL_CPU_SIZE",
        "LMCACHE_CHUNK_SIZE",
    ]:
        os.environ.pop(k, None)

    if lmcache:
        os.environ["LMCACHE_LOCAL_CPU"] = "True"
        os.environ["LMCACHE_MAX_LOCAL_CPU_SIZE"] = LMCACHE_CPU_GB
        os.environ["LMCACHE_CHUNK_SIZE"] = "256"

    prefix_flag = "" if prefix_caching else "--no-enable-prefix-caching"
    lmcache_flags = ""
    if lmcache:
        lmcache_flags = (
            f"--kv-offloading-backend lmcache "
            f"--kv-offloading-size {KV_OFFLOAD_GB} "
            f"--disable-hybrid-kv-cache-manager"
        )

    cmd = f"""
    nohup vllm serve {MODEL} \
      --host {HOST} \
      --port {PORT} \
      --generation-config vllm \
      --gpu-memory-utilization {gpu_mem_util} \
      {prefix_flag} \
      {lmcache_flags} \
      > {log_file} 2>&1 &
    """
    print(cmd)
    subprocess.run(cmd, shell=True, check=False, executable="/bin/bash")
    wait_for_server()
    show_gpu()

def start_server_native_offload(log_file, prefix_caching=True, gpu_mem_util=0.70):
    kill_vllm()
    prefix_flag = "" if prefix_caching else "--no-enable-prefix-caching"
    cmd = f"""
    nohup vllm serve {MODEL} \
      --host {HOST} \
      --port {PORT} \
      --generation-config vllm \
      --gpu-memory-utilization {gpu_mem_util} \
      {prefix_flag} \
      --kv-offloading-backend native \
      --kv-offloading-size 8 \
      > {log_file} 2>&1 &
    """
    print(cmd)
    subprocess.run(cmd, shell=True, check=False, executable="/bin/bash")
    wait_for_server()
    show_gpu()

def dataset_flags(dataset_name, cfg):
    if dataset_name == "prefix_repetition":
        return f"""
      --dataset-name prefix_repetition \
      --prefix-repetition-prefix-len {cfg['prefix_len']} \
      --prefix-repetition-suffix-len {cfg['suffix_len']} \
      --prefix-repetition-num-prefixes {cfg['num_prefixes']} \
      --prefix-repetition-output-len {cfg['output_len']} \
        """

    if dataset_name == "sharegpt":
        return f"""
      --dataset-name sharegpt \
      --dataset-path {cfg['dataset_path']} \
        """

    raise ValueError(f"Unsupported dataset: {dataset_name}")

def run_bench(result_file, dataset_name=DATASET_NAME):
    common = BENCH_CONFIG
    ds_cfg = DATASET_CONFIG[dataset_name]
    ds_flags = dataset_flags(dataset_name, ds_cfg)

    cmd = f"""
    vllm bench serve \
      --backend openai-chat \
      --model {MODEL} \
      --host {HOST} \
      --port {PORT} \
      --endpoint /v1/chat/completions \
      --num-prompts {common['num_prompts']} \
      --num-warmups {common['num_warmups']} \
      --max-concurrency {common['max_concurrency']} \
      --temperature {common['temperature']} \
      {ds_flags} \
      --save-result \
      --result-dir /content \
      --result-filename {result_file} \
      --percentile-metrics ttft,itl \
      --metric-percentiles 50,90,99
    """
    print(cmd)
    result = subprocess.run(
        cmd,
        shell=True,
        executable="/bin/bash",
        text=True,
        capture_output=True,
    )
    print("RETURN CODE:", result.returncode)
    print("\nSTDOUT:\n", result.stdout)
    print("\nSTDERR:\n", result.stderr)
    return result

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

# Condition 1 = Prefix Caching On + LMCache OFF

In [ ]:
start_server(
    log_file="cond1_no_lmcache_prefix_on.log",
    lmcache=False,
    prefix_caching=True,
    gpu_mem_util=0.3,
)

run_bench("cond1_no_lmcache_prefix_on.json")
tail_log("cond1_no_lmcache_prefix_on.log", n=120)

# Condition 2 = Prefix Caching On + LMCache ON

In [ ]:
start_server(
    log_file="cond2_lmcache_prefix_on.log",
    lmcache=True,
    prefix_caching=True,
    gpu_mem_util=0.30,
)

run_bench("sharegpt_cond2.json", dataset_name="sharegpt")
tail_log("cond2_lmcache_prefix_on.log", n=120)